# 02 – Entrenamiento YOLOv11

**Proyecto:** Detección Automática de Fracturas Óseas en Radiografías  
**Materia:** Visión por Computadora II – CEIA/FIUBA  
**Autores:** Lucia T. Capon Paul · Cesar Orellana · Leandro Britez

---

## Objetivos
- Fine-tuning de YOLOv11 sobre el dataset de fracturas óseas.
- Explorar el impacto de distintas técnicas de Data Augmentation.
- Registrar métricas por época para análisis posterior.

In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path('__file__' if '__file__' in dir() else '.').resolve()
ROOT = _NB_DIR if (_NB_DIR / 'src').exists() else _NB_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultralytics import YOLO
import yaml
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Descarga del dataset desde Roboflow

In [ ]:
# Apuntar al dataset pre-procesado que generamos con Filtros (CLAHE, Normalización...)
DATA_YAML = ROOT / 'data' / 'bone-fracture-detection-daoon-1-preprocessed' / 'data.yaml'
assert DATA_YAML.exists(), 'dataset no encontrado, ejecutá el notebook 00_preprocessing.ipynb primero.'
print(f'Dataset YAML: {DATA_YAML}')


## 2. Configuración del experimento

In [ ]:
with open(ROOT / 'configs' / 'yolov11.yaml') as f:
    cfg = yaml.safe_load(f)

print('Configuración cargada:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

## 3. Carga del modelo preentrenado

In [ ]:
model = YOLO(cfg['model_weights'])
print(f'Modelo cargado: {cfg["model_weights"]}')
print(model.info())

## 4. Fine-tuning

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=cfg['epochs'],
    imgsz=cfg['imgsz'],
    batch=cfg['batch'],
    lr0=cfg['lr0'],
    lrf=cfg['lrf'],
    momentum=cfg['momentum'],
    weight_decay=cfg['weight_decay'],
    # Data augmentation
    hsv_h=cfg['hsv_h'],
    hsv_s=cfg['hsv_s'],
    hsv_v=cfg['hsv_v'],
    flipud=cfg['flipud'],
    fliplr=cfg['fliplr'],
    mosaic=cfg['mosaic'],
    mixup=cfg['mixup'],
    # Output
    project=str(ROOT / 'results'),
    name='yolov11_fracture',
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
)

print('Entrenamiento finalizado.')
print(f'Mejor checkpoint: {results.save_dir}/weights/best.pt')

## 5. Evaluación rápida en validación

In [ ]:
val_results = model.val(data=str(DATA_YAML), split='val')
print(f'mAP@0.5:      {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'Precision:    {val_results.box.mp:.4f}')
print(f'Recall:       {val_results.box.mr:.4f}')

---
## Notas del experimento

*(Completar tras el entrenamiento)*

- **Variante de modelo usada:** ...
- **Épocas entrenadas / early stopping:** ...
- **mAP@0.5 final:** ...
- **Observaciones:** ...